# 04 — Model contract and operational scoring

**Role: Technical model review.**

This notebook verifies the saved nine-feature hurdle model, its fixed feature order, its temporal safeguards, and the annual scoring lifecycle. The retained hurdle is a continuous comparative research artifact, not a probability or buyer-facing safety score. It reads saved artefacts only; it does **not** tune, fit, score, overwrite a model, or re-present final-test performance figures.

The output of the model is an estimated next-year burned share, not a probability, safety score, or purchase recommendation.

## What this notebook is for

Use this notebook after project reproduction to inspect the modelling contract: which features are accepted, how the two-part hurdle works, which model artefacts are saved, and how the approved specification is used in the next annual scoring cycle.

The held-out final temporal-test metrics, plots, and human-readable results story appear once in [06_final_charts.ipynb](06_final_charts.ipynb). Keeping them there avoids two notebooks presenting the same evidence in different ways.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_contract import PREDICTOR_COLUMNS, TARGET_COLUMN
from src.notebook_reporting import model_component_frame
from src.notebook_support import read_json_artifact, require_artifacts, resolve_project_root

PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
metrics = read_json_artifact(PROJECT_ROOT, 'data/processed/extended_model_selection_2010_2021/final_temporal_test_metrics.json')
selection_model_path, operational_model_path = require_artifacts(PROJECT_ROOT, [
    'data/processed/extended_model_selection_2010_2021/models/nine_feature_hurdle.joblib',
    'data/processed/final_model_2010_2024/nine_feature_hurdle.joblib',
])
selection_payload = joblib.load(selection_model_path)
operational_payload = joblib.load(operational_model_path)
print('Loaded saved model artefacts and temporal contract without fitting, scoring, or displaying final-test results.')

Loaded saved model artefacts and temporal contract without fitting, scoring, or displaying final-test results.


## Feature, split, and artefact contract

The candidate specification was selected without using final-test years. The frozen final temporal test was predictor years T=2022–2024, with observed outcome years 2023–2025. This notebook reads that metadata only to verify the boundary; it does not show or reinterpret the final-test results.

The operational refit has the same approved nine-feature specification, but uses all labelled history available through outcome year 2025 before producing the separate target-free 2026 estimate.

In [2]:
assert metrics['design']['final_test_years'] == [2022, 2023, 2024]
assert metrics['design']['tuning_performed'] is False
assert selection_payload['feature_order'] == list(PREDICTOR_COLUMNS)
assert operational_payload['feature_order'] == list(PREDICTOR_COLUMNS)

feature_contract = pd.DataFrame({'feature': PREDICTOR_COLUMNS, 'role': ['predictor'] * len(PREDICTOR_COLUMNS)})
artifact_contract = pd.DataFrame([
    {
        'artifact': 'frozen final-test model',
        'predictor years used for fitting': selection_payload['train_years'] + selection_payload['validation_years'],
        'target': TARGET_COLUMN,
        'random seed': selection_payload['random_seed'],
    },
    {
        'artifact': 'current operational refit',
        'predictor years used for fitting': operational_payload['training_predictor_years'],
        'target': operational_payload['target'],
        'random seed': operational_payload['random_seed'],
    },
])
display(feature_contract)
display(artifact_contract)

,feature,role
0,built_up_share,predictor
1,forest_shrub_share_2km,predictor
2,mean_slope_2km,predictor
3,fire_years_previous_10y_2km,predictor
4,warm_season_mean_2m_temperature_c,predictor
5,warm_season_total_precipitation_mm,predictor
6,warm_season_mean_soil_water_layer1,predictor
7,warm_season_max_monthly_2m_temperature_c,predictor
8,warm_season_min_monthly_soil_water_layer1,predictor


,artifact,predictor years used for fitting,target,random seed
0,frozen final-test model,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 201...",burned_share_next_year,20260805
1,current operational refit,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 201...",burned_share_next_year,20260805


## Model structure

The hurdle model multiplies two learned quantities: estimated occurrence of any burned share and estimated burned share conditional on fire. This is appropriate for a target with many exact zeros and continuous positive values. It is not a probability model offered to a buyer.

In [3]:
component_summary = model_component_frame(selection_payload)
display(component_summary)
print('Final estimate = occurrence component × positive-share component.')
print('No feature-importance claim is shown: the saved histogram-gradient estimators have no native, directly comparable importance measure, and correlated spatial predictors require a separately designed interpretation analysis.')

,estimator,loss_or_objective,max_iter,learning_rate,max_leaf_nodes,min_samples_leaf,random_state
component,,,,,,,
occurrence,HistGradientBoostingClassifier,log_loss,120,0.08,23,120,20260805
positive-share,HistGradientBoostingRegressor,squared_error,150,0.07,23,80,20260805


Final estimate = occurrence component × positive-share component.
No feature-importance claim is shown: the saved histogram-gradient estimators have no native, directly comparable importance measure, and correlated spatial predictors require a separately designed interpretation analysis.


## Final-evaluation handoff

The model's held-out performance is deliberately presented only in [06_final_charts.ipynb](06_final_charts.ipynb), using the saved final-test prediction artefacts. That notebook contains the overall and by-year MAE/RMSE tables, positive-burn error, Capture@20%, and diagnostic plots.

This separation matters: Notebook 04 explains **what model is used and how it is governed**; Notebook 06 explains **what the final evidence says**. Neither notebook refits or retunes the model during normal review.

## Annual operational use and limits

For forecast year `Y`, build an unlabelled predictor-year `Y−1` feature matrix and score it with a model refit only through the latest labelled predictor year `Y−2`. When ICNF later supplies the observed outcome for `Y`, evaluate that saved score, add the newly labelled records, and refit the unchanged approved specification for `Y+1`.

Use `python scripts/run_project.py --mode reproduce --confirm-rebuild` for controlled full reproduction. The annual lifecycle is documented in [docs/operational_forecast_cycle.md](../docs/operational_forecast_cycle.md). Do not manually change feature order, parameters, or interpretation in this notebook.